In [ ]:
%load_ext cuml.accel
import sklearn

In [ ]:


import logging
import time
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from skimage.feature import local_binary_pattern, hog
from skimage.filters import gabor_kernel
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from tqdm import tqdm
import pickle

class Timer:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        self.start = time.time()
        return self
    def __exit__(self, *args):
        self.end = time.time()
        print(f"{self.name} Done，Used: {self.end - self.start:.2f} Seconds")

TRAIN_CSV = "data/train.csv"
TEST_CSV = "data/test.csv"
TRAIN_IMS_DIR = "data/train_ims/"
TEST_IMS_DIR = "data/test_ims/"
FEATURES_DIR = "features/"
MODEL_DIR = "models/"

for directory in [FEATURES_DIR, MODEL_DIR]:
    os.makedirs(directory, exist_ok=True)

<frozen importlib._bootstrap_external>:1181: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1181: FutureWarning: The cuda.cuda module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.driver module instead.


ModuleNotFoundError: No module named 'cuml.accel'

In [3]:
with Timer("Data Loading"):
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    train_df['image_path'] = train_df['im_name'].apply(lambda x: os.path.join(TRAIN_IMS_DIR, x))
    test_df['image_path'] = test_df['im_name'].apply(lambda x: os.path.join(TEST_IMS_DIR, x))
    print(f"train data size: {len(train_df)}, test data size: {len(test_df)}")

train data size: 50000, test data size: 10000
Data Loading Done，Used: 0.13 Seconds


In [4]:
def normalize_image(image):
    img_float = image.astype(np.float32)
    for channel in range(3):
        channel_data = img_float[:, :, channel]
        mean = np.mean(channel_data)
        std = np.std(channel_data)
        img_float[:, :, channel] = (channel_data - mean) / (std + 1e-7)
    return img_float

def apply_augmentation(image):
    augmented_images = []
    augmented_images.append(image)  # original

    # random crop + resize
    h, w = image.shape[:2]
    crop_size = int(min(h, w) * 0.875)
    h_start = np.random.randint(0, h - crop_size + 1)
    w_start = np.random.randint(0, w - crop_size + 1)
    cropped = image[h_start:h_start + crop_size, w_start:w_start + crop_size]
    cropped = cv2.resize(cropped, (32, 32))
    augmented_images.append(cropped)

    # horizontal flip
    flipped = cv2.flip(image, 1)
    augmented_images.append(flipped)

    normalized_images = [normalize_image(img) for img in augmented_images]
    return normalized_images

In [5]:
def extract_gabor_features(image):
    """Gabor features using skimage to avoid OpenCV dtype error (senior logic preserved)"""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY).astype(np.float32)
    features = []
    frequencies = [0.1, 0.25, 0.4]
    thetas = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    
    for freq in frequencies:
        for theta in thetas:
            kernel = np.real(gabor_kernel(freq, theta=theta))
            # Use scipy convolve instead of cv2.filter2D to avoid dtype error
            from scipy.ndimage import convolve
            filtered = convolve(gray, kernel, mode='reflect')
            features.extend([np.mean(filtered), np.std(filtered)])
    return features

def extract_lbp_features(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    features = []
    for radius, points in [(1,8), (2,16), (3,24)]:
        lbp = local_binary_pattern(gray, points, radius, method='uniform')
        hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, points + 3), density=True)
        features.extend(hist)
    return features

def extract_features(image):
    features = []
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # HOG multi-scale
    for cell_size in [(4,4), (8,8), (16,16)]:
        hog_feat = hog(gray, orientations=9, pixels_per_cell=cell_size,
                       cells_per_block=(2,2), block_norm='L2')
        features.extend(hog_feat)
    
    # Color moments (HSV)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    for c in range(3):
        ch = hsv[:,:,c].astype(float)
        features.extend([np.mean(ch), np.std(ch), np.max(ch),
                         np.percentile(ch, 25), np.percentile(ch, 75)])
    
    # LBP
    features.extend(extract_lbp_features(image))
    
    # Gabor
    features.extend(extract_gabor_features(image))
    
    return np.array(features)

def extract_features_with_augmentation(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return np.zeros(1970)  # fallback
    
    augmented = apply_augmentation(img)
    all_feats = []
    for aug_img in augmented:
        feats = extract_features(aug_img)
        all_feats.append(feats)
    
    # weighted average as senior did
    return np.average(all_feats, axis=0, weights=[1.0, 0.8, 0.8])

In [9]:
def extract_all_features(df, ims_dir, save_name):
    feature_file = os.path.join(FEATURES_DIR, save_name)
    if os.path.exists(feature_file):
        print(f"Loading cached {save_name}")
        with open(feature_file, 'rb') as f:
            return pickle.load(f)
    
    print(f"Extracting {save_name}...")
    feats = Parallel(n_jobs=-1)(
        delayed(extract_features_with_augmentation)(row['image_path']) 
        for _, row in tqdm(df.iterrows(), total=len(df))
    )
    feats = np.array(feats)
    labels = df['label'].values if 'label' in df.columns else None
    
    with open(feature_file, 'wb') as f:
        pickle.dump((feats, labels) if labels is not None else feats, f)
    
    return feats, labels

with Timer("Feature Extraction"):
    train_features, train_labels = extract_all_features(train_df, TRAIN_IMS_DIR, "train_features.pkl")
    test_features, _ = extract_all_features(test_df, TEST_IMS_DIR, "test_features.pkl")
    print(f"Train shape: {train_features.shape}, Test shape: {test_features.shape}")

Extracting train_features.pkl...


Extracting test_features.pkl...



























































































































































































































































































































100%|██████████| 10000/10000 [01:49<00:00, 91.45it/s]


Train shape: (50000, 2217), Test shape: (10000, 2217)
Feature Extraction Done，Used: 651.33 Seconds


In [9]:
# ==================== CELL 5: SEPARATE BLOCK - Load Features from Stored Files ====================
print("=== SEPARATE BLOCK: Loading Features from Stored Files ===")

train_feature_file = os.path.join(FEATURES_DIR, "train_features.pkl")
test_feature_file = os.path.join(FEATURES_DIR, "test_features.pkl")

# Load train features
if os.path.exists(train_feature_file):
    print(f"Loading train features from {train_feature_file}")
    with open(train_feature_file, 'rb') as f:
        data = pickle.load(f)
    if isinstance(data, tuple):
        train_features, train_labels = data
    else:
        train_features = data
        train_labels = None
    print(f"Train features loaded successfully. Shape: {train_features.shape}")
else:
    print("Train feature file not found. Will extract later.")

# Load test features
if os.path.exists(test_feature_file):
    print(f"Loading test features from {test_feature_file}")
    with open(test_feature_file, 'rb') as f:
        data = pickle.load(f)
    if isinstance(data, tuple):
        test_features, _ = data
    else:
        test_features = data
    print(f"Test features loaded successfully. Shape: {test_features.shape}")
else:
    print("Test feature file not found. Will extract later.")

=== SEPARATE BLOCK: Loading Features from Stored Files ===
Loading train features from features/train_features.pkl
Train features loaded successfully. Shape: (50000, 2217)
Loading test features from features/test_features.pkl
Test features loaded successfully. Shape: (10000, 2217)


In [ ]:
# ==================== CELL 6: Scaling + PCA + Stacking (GPU-aware) ====================
with Timer("Scaling + PCA + Stacking"):
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_features)
    test_scaled = scaler.transform(test_features)
    
    if train_scaled.shape[1] > 1000:
        pca = PCA(n_components=0.98, random_state=42)
        train_pca = pca.fit_transform(train_scaled)
        test_pca = pca.transform(test_scaled)
        print(f"After PCA: {train_pca.shape[1]} dimensions")
    else:
        print(f"Features Shape {train_scaled.shape[1]}，No PCA required")
        train_pca = train_scaled
        test_pca = test_scaled
        pca = None

    # Use GPU if available for CatBoost and XGBoost
    base_estimators = [
        ('catboost', CatBoostClassifier(
            iterations=600,
            depth=7,
            learning_rate=0.08,
            verbose=False,
            random_seed=42,
            task_type='GPU',        # Try GPU first
            thread_count=-1
        )),
        ('xgb', XGBClassifier(
            n_estimators=800,
            max_depth=7,
            learning_rate=0.07,
            tree_method='hist',      # recommended now
            device='cuda',           # this is the new way
            random_state=42
        )),
        ('rf', RandomForestClassifier(n_estimators=600, max_depth=15, n_jobs=-1, random_state=42, class_weight='balanced', max_features='log2')),
        ('et', ExtraTreesClassifier(n_estimators=700, max_depth=20, n_jobs=-1, random_state=42, class_weight='balanced', max_features='log2')),
        ('svm', SVC(kernel='poly', probability=True, random_state=42, class_weight='balanced'))
    ]

    model = StackingClassifier(
        estimators=base_estimators,
        final_estimator=XGBClassifier(n_estimators=400, max_depth=6, random_state=42, device='cuda', tree_method='hist'),
        passthrough=True,
        cv=5,
        n_jobs=-1
    )
    
    model.fit(train_pca, train_labels)
    
    joblib.dump(model, os.path.join(MODEL_DIR, "stacking_model.joblib"))
    joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.joblib"))
    if pca is not None:
        joblib.dump(pca, os.path.join(MODEL_DIR, "pca.joblib"))
    print("Model saved")

After PCA: 794 dimensions
Model saved
Scaling + PCA + Stacking Done，Used: 15667.40 Seconds


In [16]:
with Timer("Prediction"):
    predictions = model.predict(test_pca)
    test_df['label'] = predictions
    test_df[['im_name', 'label']].to_csv("submission_senior_exact.csv", index=False)
    print("Submission saved as submission_senior_exact.csv")

Submission saved as submission_senior_exact.csv
Prediction Done，Used: 81.59 Seconds
